목적: "모델에 넣을 입력 데이터 만들기"

In [3]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv('../data/processed/data.csv', parse_dates=['date'])

In [4]:
# 참가자를 4명씩 그룹으로 묶기
unique_participants = df['participant_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_participants)

group_size = 4
group_mapping = {}
for i, pid in enumerate(unique_participants):
    group_id = i // group_size + 1
    group_mapping[pid] = group_id

df['group_id'] = df['participant_id'].map(group_mapping)

In [5]:
# 걸음수, 칼로리, 운동시간, 거리의 14일 이동 평균
rolling_cols = ['daily_steps', 'calories_burned', 'duration_minutes']

for col in rolling_cols:
    df[f'{col}_14d_avg'] = df.groupby('participant_id')[col].transform(
        lambda x: x.rolling(window=14, min_periods=7).mean()
    )

# 거리 계산 (1000보 ≈ 0.8km)
df['distance_km'] = df['daily_steps'] * 0.8 / 1000
df['distance_km_14d_avg'] = df.groupby('participant_id')['distance_km'].transform(
    lambda x: x.rolling(window=14, min_periods=7).mean()
)

In [11]:
# 날짜별 그룹 평균
group_stats = df.groupby(['group_id', 'date']).agg({
    'participant_id': 'count',           # 그룹 인원
    'daily_steps_14d_avg': 'mean',       # 그룹 평균 걸음수
    'calories_burned_14d_avg': 'mean',
    'duration_minutes_14d_avg': 'mean',
    'distance_km_14d_avg': 'mean',
    'age': 'mean',                       # 평균 나이
    'bmi': 'mean',                       # 평균 BMI
    'resting_heart_rate': 'mean',        # 평균 심박수
    'season': 'first'                    # 계절
}).reset_index()

group_stats = group_stats.rename(columns={'participant_id': 'member_count'})


In [12]:
# 목표 = 현재 평균 * 인원수 * 1.1
group_stats['goal_steps'] = (
    group_stats['daily_steps_14d_avg'] *
    group_stats['member_count'] *
    1.1
).round()

# 실제 달성값 = 현재 평균 * 인원수
group_stats['actual_steps'] = (
    group_stats['daily_steps_14d_avg'] * 
    group_stats['member_count']
).round()

# 달성률 = 실제 / 목표 (노이즈 추가)
noise_factor = np.random.uniform(0.85, 1.15, len(group_stats))
group_stats['achievement_rate_steps'] = (
    group_stats['actual_steps'] / 
    group_stats['goal_steps'] * 
    noise_factor
).clip(0.3, 1.5)


In [14]:
def calculate_next_goal(current_avg, achievement_rate, member_count):
    """달성률에 따른 다음 목표 계산"""
    if achievement_rate >= 0.9:
        growth_rate = 1.12  # 12% 증가
    elif achievement_rate >= 0.7:
        growth_rate = 1.08  # 8% 증가
    elif achievement_rate >= 0.5:
        growth_rate = 1.03  # 3% 증가
    else:
        growth_rate = 0.98  # 2% 감소
    
    return current_avg * member_count * growth_rate

# 다음 목표 계산
group_stats['next_goal_steps'] = group_stats.apply(
    lambda x: calculate_next_goal(
        x['daily_steps_14d_avg'], 
        x['achievement_rate_steps'], 
        x['member_count']
    ), 
    axis=1
).round()

In [17]:
# user_min_steps 컬럼 임시 생성
group_stats['user_min_steps'] = group_stats['daily_steps_14d_avg'] * 0.6

steps_features = [
    'member_count',
    'daily_steps_14d_avg',      # ← 실제 존재하는 컬럼
    'achievement_rate_steps',
    'age',                      # group 평균이 아니라면 수정 필요
    'bmi',
    'resting_heart_rate',
    'season',
    'user_min_steps',           # 존재 여부 확인 필요
    'next_goal_steps'           # 타깃
]

# group_stats를 그대로 사용
training_steps = group_stats[steps_features].copy()

training_steps.columns = [
    'member_count',
    'group_avg_steps_14',        # 이름 바꿔도 됨
    'last_week_achievement_rate',
    'avg_age',
    'avg_bmi',
    'avg_resting_hr',
    'season',
    'user_min_steps',
    'next_goal_steps'
]

training_steps.to_csv('../data/processed/training_steps.csv', index=False)


In [ ]:
# kcal_features = [
#     'member_count',
#     'group_avg_kcal_14',
#     'achievement_rate_kcal',
#     'avg_age',
#     'avg_bmi',
#     'avg_resting_hr',
#     'season',
#     'user_min_kcal',
#     'next_goal_kcal'  # 타겟
# ]

# training_kcal = group_stats_clean[kcal_features].copy()
# training_kcal.columns = [
#     'member_count',
#     'group_avg_kcal_14',
#     'last_week_achievement_rate',  # 컬럼명 변경
#     'avg_age',
#     'avg_bmi',
#     'avg_resting_hr',
#     'season',
#     'user_min_kcal',
#     'next_goal_kcal'
# ]

# training_steps.to_csv('../data/processed/training_kcal.csv', index=False)

NameError: name 'group_stats_clean' is not defined